# One program, four surfaces — Python, the CLI, Rust and TypeScript

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.53.0/cookbook/notebooks/book/surfaces/surfaces.ipynb)

Built from [`cookbook/book/chapters/surfaces/surfaces.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/surfaces/surfaces.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.53.0 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed fixtures, on the CPU, in minutes. SCALE = "full"
# runs it over the published data and real encoders instead, on the GPU: the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.53.0", engine + "==0.53.0", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.53.0#subdirectory=cookbook/book", server + "==0.53.0"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Verbs:** `jammi sources add` · `jammi embed` · `jammi search` (the CLI) ·
`InferenceSession::add_source` / `generate_text_embeddings` / `search_by_id` (Rust) ·
`catalog.addSource` / `embedding.generateEmbeddings` / `embedding.search` (TypeScript,
`@f-inverse/jammi-client`) · `add_source` / `generate_embeddings` / `search` (Python) ·
**Checked:** parity (one program, the same rows on every surface).

Every other chapter reaches the engine from Python. Python is one of four surfaces: the
`jammi` command line, the Rust crates, and the TypeScript client drive the same engine
through the same verbs. This chapter writes one small program — register a source, embed
it, search it — and runs it on all four, then checks that they return the same rows.

The CLI, the TypeScript client and Python's remote arm talk to one server over the
`jammi.v1` wire; the Rust program links the engine in process, the way the Rust
quickstart does. So the chapter checks two properties at once: every surface exposes
the program, and the in-process engine and the server agree.

## The program and the server

The corpus is the twenty-row `tiny_corpus` and the encoder the tiny BERT checkpoint the
book ships, so every surface finishes in seconds once built. The query is by example:
the rows nearest the vector stored for row `1`. Every surface can ask it without a text
encoder of its own.

In [ ]:
import contextlib
import importlib.metadata
import json
import os
import subprocess
import tempfile
from pathlib import Path

import jammi
from jammi.testing import LiveServer
from jammi_cookbook import fixtures, programs
from jammi_cookbook.claims import claim

VERSION = importlib.metadata.version("jammi-ai")
CORPUS = fixtures.url("tiny_corpus.parquet")
MODEL = fixtures.model("tiny_bert")
ROW_KEY = "1"
work = Path(tempfile.mkdtemp(prefix="jammi_surfaces_"))

stack = contextlib.ExitStack()
server = stack.enter_context(LiveServer(work / "server"))
print(f"jammi {VERSION}; server at {server.endpoint}")
print(f"built from {'this checkout' if programs.checkout() else 'the release'}")


def rows(records) -> list[tuple[int, float]]:
    """Ranked `(id, similarity)` pairs, similarity to five places."""
    return [(int(r["id"]), round(float(r["similarity"]), 5)) for r in records]

## Python

In [ ]:
with jammi.connect(server.endpoint) as db:
    db.add_source("corpus_py", url=CORPUS, format="parquet")
    db.generate_embeddings(source="corpus_py", model=MODEL, columns=["content"], key="id")
    hits = db.search("corpus_py", row_key=ROW_KEY, k=3, select=["id", "similarity"])
python_rows = rows(hits.to_pylist())
print(python_rows)

## The command line

The `jammi` binary is a strict client of a running server. `jammi embed` and
`jammi search` are its data verbs, beside the control verbs (`sources`, `models`,
`jobs` and the rest). `search` prints its rows as JSON lines, one object per row in rank
order, so a shell pipeline or a script reads them exactly.

In [ ]:
jammi_cli = programs.cli(VERSION)


def cli(*args: str) -> str:
    return subprocess.run([jammi_cli, "--target", server.endpoint, *args],
                          capture_output=True, text=True, check=True).stdout


print(cli("sources", "add", "corpus_cli", "--url", CORPUS, "--format", "parquet"))
print(cli("embed", "corpus_cli", "--model", MODEL, "--columns", "content", "--key", "id"))
cli_out = cli("search", "corpus_cli", "--row-key", ROW_KEY, "-k", "3",
              "--select", "id,similarity")
print(cli_out)
cli_rows = rows(json.loads(line) for line in cli_out.splitlines())

## Rust

The Rust program is the Rust quickstart's shape: an `InferenceSession` over a
`JammiConfig`, the engine linked into the binary, with no server. Its artifact directory comes
from the environment, as any other setting would.

In [ ]:
print((programs.program("one_program") / "main.rs").read_text())

It builds against the crates of this release: from crates.io in a fresh runtime, or from
the workspace when the book renders in a checkout. The first build compiles the engine,
so it takes minutes; a rerun takes seconds.

In [ ]:
project = programs.rust_program(work, VERSION, "one_program")
rust_env = dict(os.environ, JAMMI_ARTIFACT_DIR=str(work / "rust"))
build = subprocess.run([programs.cargo(), "run", "-q", "--manifest-path",
                        str(project / "Cargo.toml"), "--", CORPUS, MODEL, ROW_KEY],
                       capture_output=True, text=True, env=rust_env)
assert build.returncode == 0, build.stderr[-4000:]
print(build.stdout)
rust_rows = rows(json.loads(line) for line in build.stdout.splitlines())

## TypeScript

The TypeScript client speaks gRPC-web over `fetch`, so the same script runs in Node, a
browser, or a Worker. It reads the search result, an Arrow IPC stream, with Apache
Arrow's JavaScript library.

In [ ]:
print((programs.program("one_program") / "search.ts").read_text())

In [ ]:
ts_project, ts_env = programs.ts_program(work, VERSION)
http = server.endpoint.replace("grpc://", "http://")
ts = subprocess.run([str(ts_project / "node_modules" / ".bin" / "tsx"), "search.ts",
                     http, CORPUS, MODEL, ROW_KEY],
                    cwd=ts_project, capture_output=True, text=True, env=ts_env)
assert ts.returncode == 0, ts.stderr[-4000:]
print(ts.stdout)
ts_rows = rows(json.loads(line) for line in ts.stdout.splitlines())
stack.close()

## The same rows

In [ ]:
surfaces = {"Python": python_rows, "CLI": cli_rows, "Rust": rust_rows,
            "TypeScript": ts_rows}
print(f"{'surface':<12}{'ranked (id, similarity)'}")
for name, ranked in surfaces.items():
    print(f"{name:<12}{ranked}")
agreeing = sum(ranked == python_rows for ranked in surfaces.values())
print(f"surfaces agreeing with Python: {agreeing} of {len(surfaces)}")

In [ ]:
claim("every surface returns the same neighbours", agreeing == len(surfaces),
      f"{agreeing} of {len(surfaces)} surfaces")

All four surfaces return the same three rows in the same order, with the same
similarities: row `1` itself, then its two nearest neighbours. The program is the same
three verbs in each language — register, embed, search — because every surface is a
binding of the one `jammi.v1` vocabulary, and the in-process engine behind the Rust
program is the same engine behind the server.